> <img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> **Extra.** This notebook is divelet-ml's own: it is not a section of *Dive into Deep Learning*. It follows [Section 2.5](02.05_automatic-differentiation.ipynb). Its code is adapted from [micrograd](https://github.com/karpathy/micrograd) (MIT).
> Source, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 2.5: Automatic Differentiation](02.05_automatic-differentiation.ipynb) | Next: [Section 2.6: Probability and Statistics](02.06_probability.ipynb)

# Extra: Automatic Differentiation from Scratch

[Section 2.5](02.05_automatic-differentiation.ipynb) used the
[autograd](https://github.com/HIPS/autograd) library to compute gradients, and later sections rely on it to train models.
Here we build the same machinery in a few dozen lines, on single numbers, to see what such a library does.

The idea is *reverse-mode* automatic differentiation. We run the computation forward and remember, for each result, which values it came from and how a change in
it would change its result. Then we walk back from the final result and apply the chain rule at each step.

## A Value That Remembers

A `Value` holds one number (`data`), the derivative of the final result with respect to it (`grad`), the values it was computed from, and a small function
(`_backward`) that passes its gradient on to them. Each operation defines its own local rule:
for $c = a + b$ both inputs get the gradient of $c$ unchanged,
for $c = a \cdot b$ each input gets the gradient of $c$ times the other input, and so on.
Gradients are *added* (`+=`), because a value that is used twice receives a contribution from each use.

In [ ]:
class Value:
    """A single number and its gradient (adapted from micrograd, MIT)."""

    def __init__(self, data, children=()):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only int and float powers"
        out = Value(self.data ** k, (self,))

        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(self.data, 0.0), (self,))

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # Order the values so that each comes after the ones it was computed from, then go through them in reverse.
        order, seen = [], set()

        def visit(v):
            if v not in seen:
                seen.add(v)
                for child in v._prev:
                    visit(child)
                order.append(v)
        visit(self)

        self.grad = 1.0
        for v in reversed(order):
            v._backward()

    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1
    def __repr__(self): return f"Value(data={self.data}, grad={self.grad})"

## Using It

We write an expression with ordinary Python operators. Every operator builds a new `Value` that remembers its inputs, so by the time we have `g`, it knows the whole computation.
Calling `g.backward()` then fills in `grad` for every value that `g` depends on.

In [ ]:
a = Value(-4.0)
b = Value(2.0)
c = a + b
d = a * b + b ** 3
c += c + 1
c += 1 + c + (-a)
d += d * 2 + (b + a).relu()
d += 3 * d + (b - a).relu()
e = c - d
f = e ** 2
g = f / 2.0
g += 10.0 / f
g.backward()
print(f'g = {g.data:.4f}')
print(f'dg/da = {a.grad:.4f}')
print(f'dg/db = {b.grad:.4f}')

## Checking Against autograd

Section 2.5's `grad` should give the same derivatives for the same function. With autograd the function takes `a` and `b` and returns `g`, and `grad` differentiates it with respect to one argument at a time.

In [ ]:
import autograd.numpy as np
from autograd import grad

def g_fn(a, b):
    c = a + b
    d = a * b + b ** 3
    c += c + 1
    c += 1 + c + (-a)
    d += d * 2 + np.maximum(b + a, 0)
    d += 3 * d + np.maximum(b - a, 0)
    e = c - d
    f = e ** 2
    g = f / 2.0
    g += 10.0 / f
    return g

print(f'dg/da = {grad(g_fn, 0)(-4.0, 2.0):.4f}')
print(f'dg/db = {grad(g_fn, 1)(-4.0, 2.0):.4f}')
assert np.isclose(grad(g_fn, 0)(-4.0, 2.0), a.grad) and np.isclose(grad(g_fn, 1)(-4.0, 2.0), b.grad)

The numbers agree. autograd does the same bookkeeping, but on whole arrays at once, with one rule for each NumPy function.

## Checking Numerically

A third check needs no differentiation machinery at all. By definition, the derivative of $g$ with respect to $a$ is the limit of
$$\frac{g(a + h, b) - g(a - h, b)}{2h}$$
as $h$ shrinks, so for a small $h$ we can nudge each input and watch how the output moves. This *numerical differentiation* is too slow
and too imprecise to train a network (it needs two evaluations of the whole model per parameter, and rounding errors grow as $h$ shrinks),
but it is a good test of the other two.

In [ ]:
h = 1e-6
numeric_a = (g_fn(-4.0 + h, 2.0) - g_fn(-4.0 - h, 2.0)) / (2 * h)
numeric_b = (g_fn(-4.0, 2.0 + h) - g_fn(-4.0, 2.0 - h)) / (2 * h)
print(f'dg/da = {numeric_a:.4f}')
print(f'dg/db = {numeric_b:.4f}')
assert np.isclose(numeric_a, a.grad, rtol=1e-5) and np.isclose(numeric_b, b.grad, rtol=1e-5)

## A Tiny Network

A neuron computes a weighted sum of its inputs plus a bias, and optionally applies `relu`. A layer is a list of neurons that see the same inputs, and a multilayer perceptron
([Section 5.1](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html)) stacks layers. Because all of it is built from `Value`s, the whole network is differentiable without any new rule.

In [ ]:
import random

class Neuron:
    def __init__(self, n_in, relu=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)
        self.relu = relu

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.relu() if self.relu else act

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, n_in, n_out, **kw):
        self.neurons = [Neuron(n_in, **kw) for _ in range(n_out)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, n_in, n_outs):
        sizes = [n_in] + n_outs
        self.layers = [Layer(sizes[i], sizes[i + 1], relu=i != len(n_outs) - 1) for i in range(len(n_outs))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## Training It

We give the network four examples with three inputs each and a target of $+1$ or $-1$, and minimize the sum of squared errors by gradient descent: compute the loss, reset the gradients,
call `backward()`, and move every parameter a small step against its gradient. This is the training loop of [Section 3.4](https://d2l.ai/chapter_linear-regression/linear-regression-scratch.html) with the gradients computed by our own engine.

In [ ]:
random.seed(1)
xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]

net = MLP(3, [4, 4, 1])
print(f'{len(net.parameters())} parameters')

for step in range(100):
    loss = sum((net(x) - y) ** 2 for x, y in zip(xs, ys))
    for p in net.parameters():
        p.grad = 0.0
    loss.backward()
    for p in net.parameters():
        p.data -= 0.05 * p.grad
    if step % 20 == 0 or step == 99:
        print(f'step {step:2d}  loss {loss.data:.4f}')

In [ ]:
print([round(net(x).data, 2) for x in xs], 'targets', ys)

The predictions end close to the targets. The seed matters: with `relu`, some random starts leave every neuron outputting zero, and then the loss stays at 4 (try `random.seed(0)`). Nothing here is specific to this network: any expression built from `Value`s can be differentiated, which is what lets a library train models far larger than this one.

## Exercises

1. Add an `exp` method to `Value`, with its backward rule, and check it against autograd's `np.exp`.
1. Add `tanh` and train the network with it instead of `relu`. Does the loss fall faster?
1. Call `backward()` twice on the same `g` without resetting the gradients. What happens to `a.grad`, and why?
1. Count how many `Value`s the training loss in this notebook creates for one step. Why does autograd, which works on arrays, need far fewer?

Previous: [Section 2.5: Automatic Differentiation](02.05_automatic-differentiation.ipynb) | Next: [Section 2.6: Probability and Statistics](02.06_probability.ipynb)

## Attribution and changes

This notebook is not part of *Dive into Deep Learning*. It was written for divelet-ml and refers to the book's [Section 2.5](https://d2l.ai/chapter_preliminaries/autograd.html), [3.4](https://d2l.ai/chapter_linear-regression/linear-regression-scratch.html) and [5.1](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html) by link only.
Its text and its own code are licensed under the Apache License 2.0, like divelet-ml's other code. Not affiliated with or endorsed by the book's authors.

**Code adapted from micrograd** by Andrej Karpathy (<https://github.com/karpathy/micrograd>, commit `7bc720e951fe`), MIT License:
the `Value` class and the neuron, layer and multilayer perceptron classes, and the example expression and training data.

**Changes from micrograd:**
- `Value` has no `_op` label, no `exp` or `tanh` (left as exercises), and keeps the gradient as a float. Its comments and docstrings are rewritten.
- The neural network classes are merged into one cell, without micrograd's `Module` base class; the training loop is written out in the notebook, with a fixed seed and 100 steps.
- A check against autograd (Section 2.5's library), a check by numerical differentiation and the exercises are new.

The MIT License (MIT) Copyright (c) 2020 Andrej Karpathy

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.